# N061 · 堆调度与离散事件模拟

**目标：** 区分按时间推进与逐时刻暴力模拟。

**先修：** N036, N057, N059。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 就绪队列；结束事件；空闲时间跳转；排序复合键；双堆。

## 从问题到算法

调度题区分“当前可用资源”和“未来释放事件”。单线程CPU把已到达任务按处理时间、原索引入堆；没有可运行任务时直接跳到下次到达时刻，不逐秒模拟。服务器分配用两个堆：可用堆按权重与索引排序，忙碌堆按释放时间排序。

## 最小实现

**本章接口：** `single_threaded_order(tasks)`、`assign_tasks(servers, tasks)`

In [1]:
import heapq

def single_threaded_order(tasks):
    pending=sorted((start,duration,i) for i,(start,duration) in enumerate(tasks))
    ready=[]; i=0; now=0; out=[]
    while i<len(pending) or ready:
        if not ready: now=max(now,pending[i][0])
        while i<len(pending) and pending[i][0]<=now:
            start,duration,index=pending[i]; heapq.heappush(ready,(duration,index)); i+=1
        duration,index=heapq.heappop(ready); out.append(index); now+=duration
    return out

def assign_tasks(servers,tasks):
    if not servers and tasks: raise ValueError('at least one server required')
    available=[(weight,i) for i,weight in enumerate(servers)]; heapq.heapify(available)
    busy=[]; now=0; out=[]
    for arrival,duration in enumerate(tasks):
        now=max(now,arrival)
        if not available and busy: now=max(now,busy[0][0])
        while busy and busy[0][0]<=now:
            _,weight,index=heapq.heappop(busy); heapq.heappush(available,(weight,index))
        weight,index=heapq.heappop(available); out.append(index)
        heapq.heappush(busy,(now+duration,weight,index))
    return out

## 正确性、前提与复杂度

事件时刻之间没有候选集合变化，直接跳跃不会改变决策。每次分配前释放所有已到期资源，再用完整并列规则取堆顶，与逐资源扫描定义一致。

**代价：** CPU调度O(n log n)时间/O(n)空间；服务器调度O((n+m)log m)时间/O(m)工作空间，不计输出。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tasks=[[1,2],[2,4],[3,2],[4,1]]; order=single_threaded_order(tasks)
show_table(['执行次序','任务索引','到达','处理时长'],[(j,i,*tasks[i]) for j,i in enumerate(order)])
show_table(['任务到达时刻','分配服务器'],list(enumerate(assign_tasks([3,3,2],[1,2,3,2,1,2]))))

执行次序,任务索引,到达,处理时长
0,0,1,2
1,2,3,2
2,3,4,1
3,1,2,4


任务到达时刻,分配服务器
0,2
1,2
2,0
3,2
4,1
5,2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert single_threaded_order([[1,2],[2,4],[3,2],[4,1]])==[0,2,3,1]
assert single_threaded_order([[100,2],[1000,1]])==[0,1]
assert assign_tasks([3,3,2],[1,2,3,2,1,2])==[2,2,0,2,1,2]
from random import Random
rng=Random(61)
for _ in range(200):
    weights=[rng.randrange(1,5) for _ in range(rng.randrange(1,5))]
    tasks=[rng.randrange(1,8) for _ in range(15)]; free=[0]*len(weights); now=0; ref=[]
    for arrival,duration in enumerate(tasks):
        now=max(now,arrival)
        if all(t>now for t in free): now=min(free)
        i=min((i for i,t in enumerate(free) if t<=now),key=lambda i:(weights[i],i))
        ref.append(i); free[i]=now+duration
    assert assign_tasks(weights,tasks)==ref
print('N061: 所有本章断言通过')

N061: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 构造相同到达时间的多任务。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 实现小规模逐时刻参照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1834. Single-Threaded CPU（canonical）
- 1882. Process Tasks Using Servers（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。